In [1]:
import os
from glob import glob
import pandas as pd
import numpy as np
from PIL import Image

In [2]:
# You first need to set you directory structure
# and collect the behavioral files for the localizer and the
# study task separately.  Given that each task will be modeled
# separately treat them separately.
proj_dir = '/home/eskog001/Mattfeld_PSB6351_eas'
# use os.path.join or Pathlib to define location of files
func_dir = os.path.join(proj_dir, 'behav')
# use glob and sort to grab relevant files...I would separately handle localizer and task .tsv files
loc_behav_files = sorted(glob(os.path.join(func_dir, 'sub-021_task-loc_run-*_events.tsv')))
study_behav_files = sorted(glob(os.path.join(func_dir, 'sub-021_task-study_run-*_events.tsv')))
print(f'{len(loc_behav_files)} loc files, {len(study_behav_files)} study files') # should be 2 and 4

2 loc files, 4 study files


In [3]:
# NOTE: this is an added cell to check // prints the column names + every trial_type in the
# first run of each task so the labels in the next cell can be matched to the real files
for curr_check_file in [loc_behav_files[0], study_behav_files[0]]:
    curr_check_data = pd.read_csv(curr_check_file, sep='\t')
    print(os.path.basename(curr_check_file))
    print(list(curr_check_data.columns))
    print(curr_check_data['trial_type'].value_counts(dropna=False), '\n')
    # accuracy values on the conditional trials (should be 1 / 0, maybe NaN for no response)
    if 'acc' in curr_check_data.columns:
        print(curr_check_data.loc[curr_check_data['trial_type'] == 'COND', 'acc'].value_counts(dropna=False), '\n')


sub-021_task-loc_run-1_events.tsv
['onset', 'duration', 'trial_type', 'response_time', 'stim_file', 'resp']
trial_type
face      140
math      140
scence    140
Name: count, dtype: int64 

sub-021_task-study_run-1_events.tsv
['onset', 'duration', 'trial_type', 'response_time', 'stim_file', 'resp', 'acc']
trial_type
COND        50
baseline    30
scene1      25
face1       25
scene2      25
face2       25
Name: count, dtype: int64 

acc
1.0    40
0.0    10
Name: count, dtype: int64 



In [4]:
# another new cell
# trial_type labels used in the cells below
# EDIT these to match the output of the cell above
# localizer
loc_face_label = 'face'
loc_scene_labels = ['scene', 'scence'] # scene was misspelled in the original files
# study (vCAT2) - matched as substrings of trial_type (lowercase)
study_fix_labels = ['face', 'scene']   # identifies a fixed trial (face1/face2/scene1/scene2)
study_facefix_labels = ['face']        # fixed trial whose association points to a FACE
study_scenefix_labels = ['scene']      # fixed trial whose association points to a SCENE
study_cond_labels = ['cond']           # identifies a conditional trial
study_bl_labels = ['baseline']         # identifies a perceptual baseline trial
study_acc_col = 'acc'                  # accuracy column: 1 = correct, 0 = incorrect

# Fun for checking a trial_type against a list of substring labels
def has_label(trial_type, label_list):
    return any(curr_label in str(trial_type).lower() for curr_label in label_list)

# Fun for writing AFNI timing files ; one line per run, runs separated by new lines
# (replaces the repeated with open(...) blocks in the template)
def write_evs(onset_dict, ev_file_name):
    with open(os.path.join(evs_sink_dir, ev_file_name), 'wt') as fp:
        for curr_run in sorted(onset_dict.keys()):
            # removes the square brackets -> 6.3, 30.7, 76.5
            fp.writelines([', '.join(map(str, onset_dict[curr_run])) + '\n'])
    print(f'wrote {ev_file_name}: ' + ' | '.join(f'{k}: {len([x for x in v if x != -1])}' 
                                                 for k, v in sorted(onset_dict.items())))

In [5]:
# In this cell I'm going to first work on the localizer task

# Define variables that can distinguish between runs and have 
# onset times and their duration embedded.  I used dictionaries
# This will need to be accomplished for each type of stimulus or
# or regressor you want to isolate onset times for
loc_face_onset_times = {}
loc_scene_onset_times = {}

# Iterate over your text files for the localizer task.
# The variable curr_behav_file will be a string variable with
# the full path to the separate runs of the localizer task. idx is a counter
# used for indexing.
for idx, curr_behav_file in enumerate(loc_behav_files):
    # This is where I defined the keys in my dictionaries that were created above
    # and defined the output associated with each key as an empty list
    # example --> name_of_dictionary_defined_above[f'run{SOMETHING}'] = []
    loc_face_onset_times[f'run{idx+1}'] = []
    loc_scene_onset_times[f'run{idx+1}'] = []
    
    # I'm using the pandas function read_csv to read in the log files
    curr_behav_data = pd.read_csv(curr_behav_file, sep='\t')
    
    # Given that the localizer task is a block design you don't need 
    # the onset time of each stimulus.  You just need the first and then
    # the total duration of all stimuli to create your blocked time
    # to convolve with your hemodynamic response.  I used temporary lists
    # to grab all onset times and then picked the first...you can choose
    # another way.
    tmp_face_onset = []
    tmp_scene_onset = []
    
    # iterating over ????? here...i is counter for indexing
    # What column header would you want to iterate over?  Why?
    # NOTE: trial_type - it labels what was on screen for each trial (face, scene, or math)
    for i, curr_trial_type in enumerate(curr_behav_data['trial_type']):
        if curr_trial_type == loc_face_label: #What would you want to look for in this conditional statement?
            # Here I am appending the onset of the stimulus if the current
            # trial type is a ??????.
            # I used a list append function here to grow my temporary list variable
            # reference above face-by-face...you can choose a different way
            # example --> name_of_temp_variable.append(curr_behav_data['????'][i]) #What is i representing? 
            # NOTE: i is the row (trial) index within the current run
            tmp_face_onset.append(round(curr_behav_data['onset'][i], 3))
            # NOTE: end time of the last face trial so far -> used for the block duration
            tmp_face_end = curr_behav_data['onset'][i] + curr_behav_data['duration'][i]
        elif curr_trial_type in loc_scene_labels: #note...scence was misspelled originally
            # Here I am appending the onset of the stimulus if the current
            # trial type is a scene.
            # I used a list append function here to grow my temporary list variable
            # reference above face-by-face...you can choose a different way
            tmp_scene_onset.append(round(curr_behav_data['onset'][i], 3))
            tmp_scene_end = curr_behav_data['onset'][i] + curr_behav_data['duration'][i]
            
        # here I am using the first trial type when it becomes math and the 
        # face onset list variable is ???? elements long (just exited a face block)
        # to assign the first element of the tmp_face_onset list variable to the 
        # dictionary that I created earlier.
        # the format of AFNI stimulus timing files is the following:
        #
        # onset_time:duration (e.g., 6.3:25, 12.7:25, 22.5:25) --> A block started at 6.3, 12.7, and 22.5 seconds
        # into the experiment and each block lasted for 25 seconds in length...make sure these numbers match your
        # experimental design
        # NOTE (changed from template): a block is closed as soon as ANY other trial type shows up
        # (not only math), and no fixed block length is needed. duration is computed from the data:
        # end of last trial in block - onset of first trial in block
        if curr_trial_type != loc_face_label and len(tmp_face_onset) > 0:
            # This is where I update my run dictionary with the current onset time and duration of 
            # the block of stimuli of a specific condition....HINT: 0 is the first in a list of elements
            # Be sure to reset your temporary variable as you will be now collecting a new set
            # of onset times for your new block of stimuli
            loc_face_onset_times[f'run{idx+1}'].append(f'{tmp_face_onset[0]}:{round(tmp_face_end - tmp_face_onset[0], 3)}')
            tmp_face_onset = []
        if curr_trial_type not in loc_scene_labels and len(tmp_scene_onset) > 0:
            # Do the same as you did above but for the different stimuli conditions.
            loc_scene_onset_times[f'run{idx+1}'].append(f'{tmp_scene_onset[0]}:{round(tmp_scene_end - tmp_scene_onset[0], 3)}')
            tmp_scene_onset = []
    
    # NOTE (added): close a block if the run ended inside it, otherwise the last block is dropped
    if len(tmp_face_onset) > 0:
        loc_face_onset_times[f'run{idx+1}'].append(f'{tmp_face_onset[0]}:{round(tmp_face_end - tmp_face_onset[0], 3)}')
    if len(tmp_scene_onset) > 0:
        loc_scene_onset_times[f'run{idx+1}'].append(f'{tmp_scene_onset[0]}:{round(tmp_scene_end - tmp_scene_onset[0], 3)}')
            
# The following code creates a string element that has the square brackets
# removed.  This is important for the following steps below.
# I'm leaving this for you...If you opt to not use dictionaries like I did then
# you'll have to find a different way.
# NOTE: the ", ".join(...) is done inside write_evs() for every run

# Here I am defining the sink directory where I would like to save the timing files
# HINT you shouldn't save these text files in teh same directory where your code resides
# you can also you Pathlib to create this and make the direcotry if it doesn't exist
evs_sink_dir = os.path.join(proj_dir, 'derivatives', 'first_lvl', 'sub-021', 'evs')
# I check to see if the directory exists.  If it doesn't I create it.
if not os.path.isdir(evs_sink_dir):
    os.makedirs(evs_sink_dir)
    
# below I am defining the file names for the localizer (loc) face and scene evs.
# each run is captured on a separate line with the multiple onsets within a run
# captured on a single line
# I'm leaving this for you...again this may constrain the way you do things above
# but it gives you a hint of how to output data into a AFNI style format
# your output should eventually look like the following:
#
# 6.3:25, 30.7:25, 76.5:25, 137.3:25, 225.1:25
# 6.3:25, 30.7:25, 76.5:25, 137.3:25, 225.1:25
#
# The first line refers to run1 and the second line refers to run2
# in this case the onset times were identical for this particular condition across runs
write_evs(loc_scene_onset_times, 'loc_scene_evs.1D')
write_evs(loc_face_onset_times, 'loc_face_evs.1D')

wrote loc_scene_evs.1D: run1: 7 | run2: 7
wrote loc_face_evs.1D: run1: 7 | run2: 7


In [6]:
# Similar to above I am creating empty dictionary variables
# for each of the events that I am interested in.
# I will then insert run keys to separate the timing files 
# for the events of interest and their specific runs.
task_fixb4_c_cond_onset_times = {}   # fixed trial before a CORRECT conditional trial
task_fixb4_ic_cond_onset_times = {}  # fixed trial before an INCORRECT conditional trial
task_facefixb4_bl_onset_times = {}   # face fixed trial before a baseline trial (MVPA)
task_scenefixb4_bl_onset_times = {}  # scene fixed trial before a baseline trial (MVPA)
task_remain_events_onset_times = {}  # all remaining task events
study_ev_dicts = [task_fixb4_c_cond_onset_times, task_fixb4_ic_cond_onset_times,
                  task_facefixb4_bl_onset_times, task_scenefixb4_bl_onset_times,
                  task_remain_events_onset_times]

# Here I am iterating over the study behavior files.  There should be
# 4 of them.
for idx, curr_behav_file in enumerate(study_behav_files):
    # I set the run key for each condition of interest
    # Again if you opt not to use the dictionaries as I did
    # you'll need to utilize a different way. HINT you should
    # update the key for as many events of interest you are
    # attempting to isolate
    for curr_dict in study_ev_dicts:
        curr_dict[f'run{idx+1}'] = []
    
    # I read in the current study run behavioral file
    # NOTE: sometimes it is not good practice to use the same
    # variable names across cells. It may execute but erroneously
    curr_behav_data = pd.read_csv(curr_behav_file, sep='\t')
    
    # I iterate now over the contents of the run specific data.
    # Again...you'll want to isolate the column header that lets
    # you assess the event or trial types of interest
    for i, curr_trial_type in enumerate(curr_behav_data['trial_type']):
        # NOTE (added): label of the previous trial - empty on the first trial so that
        # [i-1] never reads row -1 (pandas raises a KeyError there)
        prev_trial_type = curr_behav_data['trial_type'][i-1] if i > 0 else ''
        prev_is_fix = has_label(prev_trial_type, study_fix_labels)
        
        # I am evaluating whether or not the current trail type was a 
        # fixed association that had a conditional trial that followed with a face
        # or a scene
        if has_label(curr_trial_type, study_fix_labels):
            # if it was either of those grab that onset
            tmp_fix_onset = round(curr_behav_data['onset'][i], 3)
            # if this is not our first trial (i = counter > 0) - remember python is 0-based
            if i > 0:
                # evaluate whether or not the LAST TRIAL (i-1) was a scene or face fix trial
                # grab the current onset time and assign it to the remaining events.
                # In the analysis that I am interested in pursuing I want to separate these
                # trials from trials where the fix face and scence trials are followed either
                # by a conditional trial or by a baseline trial
                # NOTE (changed from template): uses onset[i-1], not onset[i]. the PREVIOUS fixed
                # trial is the one followed by another fixed trial. the current one is still held
                # in tmp_fix_onset and may go to a fix_b4 regressor -> [i] would count it twice
                if prev_is_fix:
                    task_remain_events_onset_times[f'run{idx+1}'].append(round(curr_behav_data['onset'][i-1], 3))
        # Here I am evaluating whether or not the current trial type is a conditional trial 
        # that was responded to correctly
        elif has_label(curr_trial_type, study_cond_labels) and prev_is_fix and curr_behav_data[study_acc_col][i] == 1:
            task_fixb4_c_cond_onset_times[f'run{idx+1}'].append(tmp_fix_onset)
        # grab the onset times for fixed association trials before correct and incorrect conditional trials
        # this will require multiple elif statements.  You'll have to assess what the curr_trial_type is
        # and whether or not that trial was correct or incorrect.  You'll then input that 'tmp_fix_onset'
        # variable that you held onto when you evaluated the above trial types
        elif has_label(curr_trial_type, study_cond_labels) and prev_is_fix and curr_behav_data[study_acc_col][i] == 0:
            task_fixb4_ic_cond_onset_times[f'run{idx+1}'].append(tmp_fix_onset)
        # Now I am doing the same thing for trials that precede the perceptual baseline trials
        # These trials will be used for the MVPA anlaysis that is planned.
        elif has_label(curr_trial_type, study_bl_labels) and prev_is_fix and has_label(prev_trial_type, study_facefix_labels):
                task_facefixb4_bl_onset_times[f'run{idx+1}'].append(tmp_fix_onset)
        elif has_label(curr_trial_type, study_bl_labels) and prev_is_fix and has_label(prev_trial_type, study_scenefix_labels):
                task_scenefixb4_bl_onset_times[f'run{idx+1}'].append(tmp_fix_onset)
        
        # NOTE (added): the conditional trial itself is task activity -> remaining events
        # (not modeled in the template, so it would have fallen into the baseline)
        # a fixed trial before a no-response conditional trial (acc not 0 or 1) also goes here
        if has_label(curr_trial_type, study_cond_labels):
            task_remain_events_onset_times[f'run{idx+1}'].append(round(curr_behav_data['onset'][i], 3))
            if prev_is_fix and curr_behav_data[study_acc_col][i] not in [0, 1]:
                task_remain_events_onset_times[f'run{idx+1}'].append(tmp_fix_onset)
    
    # NOTE (added): run ended on a fixed trial -> remaining events
    if has_label(curr_behav_data['trial_type'].iloc[-1], study_fix_labels):
        task_remain_events_onset_times[f'run{idx+1}'].append(tmp_fix_onset)
    # NOTE (added): keep each run's onsets in time order (remaining events are added out of order)
    for curr_dict in study_ev_dicts:
        curr_dict[f'run{idx+1}'].sort()
            
# Given that we're setting things up to analyze in AFNI
# you can't have runs that don't have any events in them....or
# you can but you can't have an empty row...thus here I am checking
# to see if the runs are empty and if they are adding a filler (-1)
for curr_run in ['run1', 'run2', 'run3', 'run4']:
    for curr_dict in study_ev_dicts:
        if len(curr_dict[curr_run]) == 0:
            curr_dict[curr_run].append(-1)
    # the code above would have to be repeated for each variable that is 
    # capturing a stimulus condition of interest. THis is not the most
    # effecient way of coding and can be improved upon.
    # NOTE: looping over study_ev_dicts handles every condition at once

# Reformatting as before to save as a text file with no square brackets
# use what you did for the localizer task here. If you use the same way as before
# or that I used you'll have to have a separate line for each run of each condition
# This is not a great way to code and could be improved upon
# NOTE: handled inside write_evs()

# Creating and checking to see if the directory exists for where you want to save your text files
# use the way you did this for the localizer task
if not os.path.isdir(evs_sink_dir):
    os.makedirs(evs_sink_dir)

# creating my separate ev files with runs written to each line
# use the way you did this before. HINT if you keep the same way as before
# and how I did it you'll need a separate block of code for each regressor (aka stimulus condition)
# you are trying to model and a separate line within each block for each run
# In the end it should look something like the following in one file:
#
# 6.3, 30.7, 76.5, 137.3, 225.1
# 6.3, 30.7, 76.5, 137.3, 225.1
# 6.3, 30.7, 76.5, 137.3, 225.1
# 6.3, 30.7, 76.5, 137.3, 225.1
# 
# In the example above this stimulus had the exact same onset times across each runs
# your output should in format look like this but your onset times should not be the
# same across runs!!!!
write_evs(task_fixb4_c_cond_onset_times, 'fix_b4_c_cond_evs.1D')
write_evs(task_fixb4_ic_cond_onset_times, 'fix_b4_ic_cond_evs.1D')
write_evs(task_remain_events_onset_times, 'events_remain_evs.1D')
write_evs(task_facefixb4_bl_onset_times, 'facefix_b4_bl_evs.1D')
write_evs(task_scenefixb4_bl_onset_times, 'scenefix_b4_bl_evs.1D')


wrote fix_b4_c_cond_evs.1D: run1: 40 | run2: 48 | run3: 40 | run4: 50
wrote fix_b4_ic_cond_evs.1D: run1: 10 | run2: 2 | run3: 10 | run4: 0
wrote events_remain_evs.1D: run1: 70 | run2: 70 | run3: 70 | run4: 70
wrote facefix_b4_bl_evs.1D: run1: 10 | run2: 19 | run3: 15 | run4: 14
wrote scenefix_b4_bl_evs.1D: run1: 20 | run2: 11 | run3: 15 | run4: 16


In [7]:
# This cell is required to run the following 3dDeconvolve calls in the subsequent cells
# Here I am collecting the motion related regressor files that were created for each run
# separately, concatenating them and saving the output.

# the below directory structure will change to match how your data are organized
# NOTE: datasink layout from preproc3.py -> derivatives/preproc/sub-021/motion/_volreg*/<run>_bold.1D
motion_dir = os.path.join(proj_dir, 'derivatives', 'preproc', 'sub-021', 'motion')

# this is new (added): only concatenate if every run has a motion file. my preproc5.py currently only
# processes study run-1, so otherwise the design matrices are built without motion
loc_motion_files = sorted(glob(os.path.join(motion_dir, '*', '*task-loc*bold.1D')))
if len(loc_motion_files) == 2:
    loc_mot_dict = {}
    for mot_i, curr_loc_motion_file in enumerate(loc_motion_files):
        loc_mot_dict[f'run{mot_i + 1}'] = np.genfromtxt(curr_loc_motion_file)
        
    allruns_loc_motion_data = np.concatenate((loc_mot_dict['run1'], loc_mot_dict['run2']))
    np.savetxt(os.path.join(motion_dir, 'allruns_loc_mot_data.1D'), allruns_loc_motion_data)
else:
    print(f'found {len(loc_motion_files)} of 2 loc motion files -> loc design matrix built WITHOUT motion')

task_motion_files = sorted(glob(os.path.join(motion_dir, '*', '*task-study*bold.1D')))
if len(task_motion_files) == 4:
    task_mot_dict = {}
    for mot_i, curr_task_motion_file in enumerate(task_motion_files):
        task_mot_dict[f'run{mot_i + 1}'] = np.genfromtxt(curr_task_motion_file)
        
    allruns_task_motion_data = np.concatenate((task_mot_dict['run1'], task_mot_dict['run2'],
                                               task_mot_dict['run3'], task_mot_dict['run4']))
    np.savetxt(os.path.join(motion_dir, 'allruns_task_mot_data.1D'), allruns_task_motion_data)
else:
    print(f'found {len(task_motion_files)} of 4 study motion files -> study design matrix built WITHOUT motion')

found 0 of 2 loc motion files -> loc design matrix built WITHOUT motion
found 1 of 4 study motion files -> study design matrix built WITHOUT motion


In [8]:
%%bash
# Given that we are running this with no data (see -nodata flag) we can run a quick bash
# command in the cell by using line above.  The matrices that are created and image
# will be created in the directory where this jupyter notebook is running. In my case.
# /home/data/madlab/Mattfeld_PSB6351/mattfeld_2020/code

# first load AFNI and set dirs
export PATH=$PATH:/home/applications/afni/25.2.07
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/motif-2.3.8-5gv6dfpysx6x42sq5ahtrb5xbvdvitdf/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxt-1.3.1-io2ikvkn4dc5owt2whr5k7synmsmpyii/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxft-2.3.8-dj2jatifobe67j534lbw3dnqqfn6olzg/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxmu-1.2.1-2iyuzberhuplw6zkrfppayebxfwpde6x/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libsm-1.2.6-bfopuwkxowogrmgea22q3jog4m3icmbo/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libice-1.1.2-7dyepgav4bh335ib7miuy3uar2f5zoui/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libpng-1.6.47-hshbhaybhl3dg7pojxgaplot2djkwln2/lib64:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxext-1.3.6-s2m6nnzuwvo2dvdjqby2lgmvgoebnebe/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libx11-1.8.11-dsqyqveqbcsztzcr5hou4vupdcnverqr/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxpm-3.5.17-7tbkl7pprcdkknfzfd6zuvlpxeobuqif/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/mesa-25.0.5-ii4yqltj6qjf7iisbyyve5v4fgbqwcbg/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/mesa-glu-9.0.2-evnqwufx3z5anlf3sdb6uiormskw7qyz/lib:/home/applications/afni/25.2.07/lib

proj_dir=/home/eskog001/Mattfeld_PSB6351_eas
evs_dir=${proj_dir}/derivatives/first_lvl/sub-021/evs
mot_file=${proj_dir}/derivatives/preproc/sub-021/motion/allruns_loc_mot_data.1D

# -concat gets the starting volume of each run
nt_total=608 #loc: 2 x 304 = 608, study: 4 x 355 = 1420
concat="1D: 0 304"

# only include motion if the concatenated file exists
ortvec_opt=""
if [ -f ${mot_file} ]; then ortvec_opt="-ortvec ${mot_file} motion"; fi

# -nodata: build the design matrix only (no fMRI data needed)
# dmBLOCK(1) + onset:duration in the ev files -> each block convolved at its own length
3dDeconvolve -nodata ${nt_total} 1.76 \
-concat "${concat}" \
${ortvec_opt} \
-polort A \
-local_times \
-num_stimts 2 \
-stim_times_AM1 1 ${evs_dir}/loc_face_evs.1D "dmBLOCK(1)" -stim_label 1 faces \
-stim_times_AM1 2 ${evs_dir}/loc_scene_evs.1D "dmBLOCK(1)" -stim_label 2 scenes \
-x1D X.loc.xmat.1D -xjpeg X.loc.jpg


++ '-stim_times_AM1 1 /home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_face_evs.1D' has 1 auxiliary values per time point
++ '-stim_times_AM1 1': basis function model 'dmBLOCK(1)' uses 1 parameters,
    out of the 1 found in timing file '/home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_face_evs.1D'
++ '-stim_times_AM1 2 /home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_scene_evs.1D' has 1 auxiliary values per time point
++ '-stim_times_AM1 2': basis function model 'dmBLOCK(1)' uses 1 parameters,
    out of the 1 found in timing file '/home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_scene_evs.1D'
++ 3dDeconvolve: AFNI version=AFNI_25.2.07 (Aug  6 2025) [64-bit]
++ Authored by: B. Douglas Ward, et al.
++ using TR=1.76 seconds for -stim_times and -nodata
++ using NT=608 time points for -nodata
++ Imaging duration=535.0 s; Automatic polort=4
++ -stim_times using TR=1.76 s for stimulus timing 


Stimulus: faces 
  h[ 0] norm. std. dev. =   0.1137

Stimulus: scenes 
  h[ 0] norm. std. dev. =   0.1137


In [11]:
%%bash
# Given that we are running this with no data (see -nodata flag) we can run a quick bash
# command in the cell by using line above.  The matrices that are created and image
# will be created in the directory where this jupyter notebook is running. In my case.
# /home/data/madlab/Mattfeld_PSB6351/mattfeld_2020/code

# AFNI paths (same as estimate_smoothness_ext.sh)
export PATH=$PATH:/home/applications/afni/25.2.07
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/motif-2.3.8-5gv6dfpysx6x42sq5ahtrb5xbvdvitdf/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxt-1.3.1-io2ikvkn4dc5owt2whr5k7synmsmpyii/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxft-2.3.8-dj2jatifobe67j534lbw3dnqqfn6olzg/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxmu-1.2.1-2iyuzberhuplw6zkrfppayebxfwpde6x/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libsm-1.2.6-bfopuwkxowogrmgea22q3jog4m3icmbo/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libice-1.1.2-7dyepgav4bh335ib7miuy3uar2f5zoui/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libpng-1.6.47-hshbhaybhl3dg7pojxgaplot2djkwln2/lib64:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxext-1.3.6-s2m6nnzuwvo2dvdjqby2lgmvgoebnebe/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libx11-1.8.11-dsqyqveqbcsztzcr5hou4vupdcnverqr/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/libxpm-3.5.17-7tbkl7pprcdkknfzfd6zuvlpxeobuqif/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/mesa-25.0.5-ii4yqltj6qjf7iisbyyve5v4fgbqwcbg/lib:/home/applications/spack-202506/opt/spack/linux-sapphirerapids/mesa-glu-9.0.2-evnqwufx3z5anlf3sdb6uiormskw7qyz/lib:/home/applications/afni/25.2.07/lib

proj_dir=/home/eskog001/Mattfeld_PSB6351_eas
evs_dir=${proj_dir}/derivatives/first_lvl/sub-021/evs
mot_file=${proj_dir}/derivatives/preproc/sub-021/motion/allruns_loc_mot_data.1D

# -concat gets the starting volume of each run
nt_total=608
concat="1D: 0 304"

# NOTE (added): only include motion if the concatenated file exists
ortvec_opt=""
if [ -f ${mot_file} ]; then ortvec_opt="-ortvec ${mot_file} motion"; fi

3dDeconvolve -nodata ${nt_total} 1.76 \
-concat "${concat}" \
${ortvec_opt} \
-polort A \
-local_times \
-num_stimts 2 \
-stim_times_AM1 1 ${evs_dir}/loc_face_evs.1D "dmBLOCK(1)" -stim_label 1 faces \
-stim_times_AM1 2 ${evs_dir}/loc_scene_evs.1D "dmBLOCK(1)" -stim_label 2 scenes \
-x1D X.loc.xmat.1D -xjpeg X.loc.jpg


++ '-stim_times_AM1 1 /home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_face_evs.1D' has 1 auxiliary values per time point
++ '-stim_times_AM1 1': basis function model 'dmBLOCK(1)' uses 1 parameters,
    out of the 1 found in timing file '/home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_face_evs.1D'
++ '-stim_times_AM1 2 /home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_scene_evs.1D' has 1 auxiliary values per time point
++ '-stim_times_AM1 2': basis function model 'dmBLOCK(1)' uses 1 parameters,
    out of the 1 found in timing file '/home/eskog001/Mattfeld_PSB6351_eas/derivatives/first_lvl/sub-021/evs/loc_scene_evs.1D'
++ 3dDeconvolve: AFNI version=AFNI_25.2.07 (Aug  6 2025) [64-bit]
++ Authored by: B. Douglas Ward, et al.
++ using TR=1.76 seconds for -stim_times and -nodata
++ using NT=608 time points for -nodata
++ Imaging duration=535.0 s; Automatic polort=4
++ -stim_times using TR=1.76 s for stimulus timing 


Stimulus: faces 
  h[ 0] norm. std. dev. =   0.1137

Stimulus: scenes 
  h[ 0] norm. std. dev. =   0.1137


In [12]:
im_task = Image.open(os.path.join(os.getcwd(), 'X.task.jpg'))
im_task

FileNotFoundError: [Errno 2] No such file or directory: '/home/eskog001/Mattfeld_PSB6351_eas/code/conversion/X.task.jpg'

In [ ]:
im_loc = Image.open(os.path.join(os.getcwd(), 'X.loc.jpg'))
im_loc

In [13]:
!ls -l X.*

-rw-r--r-- 1 eskog001 hpc_classroom 245134 Nov  8 20:01 X.loc.jpg
-rw-r--r-- 1 eskog001 hpc_classroom  63666 Nov  8 20:01 X.loc.xmat.1D
-rw-r--r-- 1 eskog001 hpc_classroom   2172 Nov  8 20:01 X.loc_XtXinv.xmat.1D
